In [0]:
%pip install 'databricks-sdk>=0.118.0' sentence-transformers trafilatura requests pandas

In [0]:
%restart_python

In [0]:
import base64
from urllib.parse import urlparse

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def get_lakebase_url() -> str:
    secret = w.secrets.get_secret(scope="database", key="lakebase-url")
    return base64.b64decode(secret.value).decode("utf-8")


lakebase_url = get_lakebase_url()
parsed = urlparse(lakebase_url)

# Extract connection details directly from the secret URL
db_host = parsed.hostname
db_port = parsed.port or 5432
db_name = parsed.path.lstrip('/')
db_user = parsed.username
db_password = parsed.password

print(f"Connection details:")
print(f"  Host: {db_host}:{db_port}")
print(f"  Database: {db_name}")
print(f"  User: {db_user}")
print(f"  Using raw credentials from secret (no OAuth)")

In [0]:
dbutils.widgets.text("watchlist_table_name", "watchlist_tickers", "Source table (watchlist symbols)")
dbutils.widgets.text("embedding_model", "sentence-transformers/all-MiniLM-L6-v2", "Embedding model")
dbutils.widgets.text("news_lookback_days", "7", "News lookback window (days)")
dbutils.widgets.text("news_fetch_limit", "10", "Max articles to fetch per ticker")

WATCHLIST_TABLE_NAME = dbutils.widgets.get("watchlist_table_name")
EMBEDDING_MODEL_NAME = dbutils.widgets.get("embedding_model")
NEWS_LOOKBACK_DAYS = int(dbutils.widgets.get("news_lookback_days"))
NEWS_FETCH_LIMIT = int(dbutils.widgets.get("news_fetch_limit"))

In [0]:
from massive_client import MassiveClient

import base64 as _b64
import json as _json
import time


import psycopg2
import requests

client = MassiveClient()

def get_watchlist_tickers() -> list[str]:
    """Distinct, uppercased ticker symbols currently tracked across all users
    in the watchlist table - these are the only tickers we fetch news for."""
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        sslmode='require'
    )
    try:
        cursor = conn.cursor()
        cursor.execute(f"SELECT DISTINCT ticker FROM {WATCHLIST_TABLE_NAME} WHERE ticker IS NOT NULL")
        symbols = cursor.fetchall()
        return [row[0].strip().upper() for row in symbols if row[0]]
    finally:
        cursor.close()
        conn.close()

tickers = get_watchlist_tickers()
print(tickers)


In [0]:
from datetime import datetime, timezone, timedelta
import pandas as pd
price_rows = []
for ticker in tickers:
    try:
        resp = client.get_latest_price(ticker)
        result = resp["results"][0]
        snapshot_date = datetime.fromtimestamp(result["t"]/1000, tz=timezone.utc).date()
        price_rows.append(
            {
                "ticker": result["T"],
                "snapshot_date": snapshot_date,
                "open_price": result.get("o"),
                "close_price": result.get("c"),
                "high_price": result.get("h"),
                "low_price": result.get("l"),
                "volume": int(result.get("v", 0))
            }
        )
    except Exception as e:
        print(f"Price fetch failed for {ticker}:{e}")
price_df = pd.DataFrame(price_rows)
display(price_df)

In [0]:
import importlib
import massive_client
importlib.reload(massive_client)
from massive_client import MassiveClient

conn = psycopg2.connect(
    host=db_host,
    port=db_port,
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode='require'
)
client = MassiveClient()
company_rows = []
for ticker in tickers:
    try:
        details = client.get_ticker_details(ticker)
        company_rows.append(
            {
                "ticker": details.get("ticker", ticker),
                "company_name": details.get("name"),
                "description": details.get("description"),
                "sic_code": details.get("sic_code"),
                "sic_description": details.get("sic_description"),
                "market_cap": details.get("market_cap"),
                "homepage_url": details.get("homepage_url"),
                "primary_exchange": details.get("primary_exchange"),
                "total_employees": details.get("total_employees"),
                "list_date": details.get("list_date"),  # already YYYY-MM-DD string
            }
        )
    except Exception as e:
        print(f"Company details fetch failed for {ticker}: {e}")
 
company_df = pd.DataFrame(company_rows)

if len(company_df) > 0:
    display(company_df)
 
    # --- Write companies (upsert on ticker) ---
    with conn.cursor() as cur:
        for row in company_df.to_dict(orient="records"):
            cur.execute(
                """
                INSERT INTO companies
                    (ticker, company_name, description, sic_code, sic_description,
                     market_cap, homepage_url, primary_exchange, total_employees, list_date)
                VALUES (%(ticker)s, %(company_name)s, %(description)s, %(sic_code)s, %(sic_description)s,
                        %(market_cap)s, %(homepage_url)s, %(primary_exchange)s, %(total_employees)s, %(list_date)s)
                ON CONFLICT (ticker) DO UPDATE SET
                    company_name = EXCLUDED.company_name,
                    description = EXCLUDED.description,
                    sic_code = EXCLUDED.sic_code,
                    sic_description = EXCLUDED.sic_description,
                    market_cap = EXCLUDED.market_cap,
                    homepage_url = EXCLUDED.homepage_url,
                    primary_exchange = EXCLUDED.primary_exchange,
                    total_employees = EXCLUDED.total_employees,
                    list_date = EXCLUDED.list_date,
                    updated_at = NOW()
                """,
                row,
            )
    conn.commit()
    print(f"Wrote {len(company_df)} company rows.")
else:
    print("No company data fetched. Skipping write.")

conn.close()


In [0]:
import trafilatura

lookback_date = (
    datetime.now(tz=timezone.utc) - timedelta(days = NEWS_LOOKBACK_DAYS)
).strftime("%Y-%m-%d")
news_rows = []
for ticker in tickers:
    try:
        articles = client.get_news(ticker, limit=NEWS_FETCH_LIMIT, published_utc_gte=lookback_date)
    except Exception as e:
        print(f"News fetch failed for {ticker}:{e}")
        continue

    for article in articles:
        body_text = None
        url = article.get("article_url")
        if url:
            try:
                downloaded = trafilatura.fetch_url(url)
                if downloaded:
                    body_text = trafilatura.extract(downloaded)
            except Exception as e:
                print(f"Scrape failed for {url}: {e}")
        if not body_text:
            body_text = article.get("description") or article.get("title") or ""
        if not body_text.strip():
            continue

        news_rows.append(
            {
                "ticker": ticker,
                "headline": article.get("title", "")[:500],
                "body_text": body_text,
                "source":article.get("publisher", {}).get("name") if isinstance(article.get("publisher"), dict) else article.get("publisher"),
                "published_at": article.get("published_utc")
            }
        )
news_df = pd.DataFrame(news_rows)
if len(news_df) > 0:
    display(news_df)
else:
    print(f"No news articles fetched. Total tickers attempted: {len(tickers)}")
              

In [0]:
conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        sslmode='require'
    )
with conn.cursor() as cur:
    for row in price_df.to_dict(orient="records"):
        cur.execute(
            """
            INSERT INTO price_snapshots
                (ticker, snapshot_date, open_price, close_price, high_price, low_price, volume)
            VALUES (%(ticker)s, %(snapshot_date)s, %(open_price)s, %(close_price)s, %(high_price)s, %(low_price)s, %(volume)s)
            ON CONFLICT (ticker, snapshot_date) DO UPDATE SET
                open_price = EXCLUDED.open_price,
                close_price = EXCLUDED.close_price,
                high_price = EXCLUDED.high_price,
                low_price = EXCLUDED.low_price,
                volume = EXCLUDED.volume
            """,
            row,
        )
conn.commit()
print(f"Wrote {len(price_df)} price rows.")

In [0]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def chunk_text(text: str, chunk_size: int = 180, overlap: int = 40) -> list[str]:
    """
    Word-based sliding-window chunker. chunk_size=180 words stays safely
    under all-MiniLM-L6-v2's ~256 word-piece limit even for text with
    longer words/punctuation. overlap=40 keeps context continuous across
    chunk boundaries so a fact split across two chunks isn't lost to search.
    """
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start = end - overlap
    return chunks

conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        dbname=db_name,
        user=db_user,
        password=db_password,
        sslmode='require'
    )
    
with conn.cursor() as cur:
    for row in news_df.to_dict(orient="records"):
        cur.execute(
            """
            INSERT INTO news_articles (ticker, headline, body_text, source, published_at )
            VALUES (%(ticker)s, %(headline)s, %(body_text)s, %(source)s, %(published_at)s)
            ON CONFLICT (ticker, published_at) DO UPDATE SET
                headline = EXCLUDED.headline,
                body_text = EXCLUDED.body_text,
                source = EXCLUDED.source
            RETURNING article_id
            """,
            row,
        )
        article_id = cur.fetchone()[0]
        chunks = chunk_text(row["body_text"])
        if not chunks:
            continue
        # Delete existing chunks for this article before inserting new ones
        cur.execute("DELETE FROM article_chunks WHERE article_id = %s", (article_id,))
        
        chunk_embeddings = model.encode(chunks, show_progress_bar=True)
        for idx, (chunk, emb) in enumerate(zip(chunks, chunk_embeddings)):
            cur.execute(
              """
              Insert into article_chunks(article_id, chunk_index, chunk_text, embedding)
              Values(%s, %s, %s, %s)
              """,
              (article_id, idx, chunk, emb.tolist())
            )

conn.commit()
print(f"Wrote {len(news_df)} articles and there chunks.")
 
conn.close()